# 🏁 Lab: Build a Production Evaluation Pipeline

## Module 08 — End-to-End Evaluation Pipeline (Capstone) 🚀

---

## Overview

Congratulations — you've made it to the **capstone lab**! In this final module, you'll bring together **everything from Modules 01–07** to build a production-ready evaluation pipeline for a multi-component AI system. The system combines:

- **Structured output extraction** (entity extraction from contracts)
- **RAG-based document Q&A** (retrieving and answering questions from a knowledge base)
- **Free-form text generation** (generating executive summaries)

You will design evaluation stages for CI, pre-production, and production; implement the pipeline; create a readiness scorecard; compare two model versions; and make a concrete launch-readiness recommendation.

---

### 🔨 What You'll Build

By the end of this lab you will have built an **end-to-end evaluation pipeline** that follows this flow:

```
┌──────────────┐    ┌─────────────────────┐    ┌─────────────────────┐
│  CI Checks   │───▶│  Pre-Production Eval │───▶│ Readiness Scorecard │
│  (fast gate) │    │  (thorough suite)    │    │ (go/no-go summary)  │
└──────────────┘    └─────────────────────┘    └─────────────────────┘
                                                         │
                    ┌─────────────────────┐              │
                    │ Production Monitor  │◀─────────────┘
                    │ (live traffic)      │     (if approved)
                    └─────────────────────┘
                              │
                    ┌─────────────────────┐
                    │  Regression Detect  │
                    │  (v2.0 vs v2.1)     │
                    └─────────────────────┘
                              │
                    ┌─────────────────────┐
                    │  Launch Decision    │
                    │  (evidence-based)   │
                    └─────────────────────┘
```

Specifically, you will have:
- ✅ Explored evaluation data for a multi-component AI system
- ✅ Designed multi-stage evaluation with appropriate metrics per stage
- ✅ Defined gating criteria for CI → pre-production → production
- ✅ Built a readiness scorecard template
- ✅ Compared evaluation results across two model versions
- ✅ Detected and interpreted regressions (both obvious and subtle)
- ✅ Made a launch-readiness decision under ambiguity

> 💡 **Key Insight:** The goal of an evaluation pipeline is not perfection — it's **informed confidence** in launch decisions. You're building the evidence trail that lets decision-makers say "yes" or "not yet" with data, not gut feeling.

## Learning Objectives

1. Design multi-stage evaluation workflows (CI, pre-production, production)
2. Define gating criteria for model deployment stages
3. Build pre-launch readiness scorecards
4. Implement regression detection across model versions
5. Make concrete launch-readiness decisions based on evidence

## Prerequisites

- Modules 01–07 completed
- Familiarity with evaluation metrics (precision, recall, faithfulness, toxicity, etc.)
- Python proficiency (pandas, numpy, dictionaries, basic stats)

## Estimated Time

**20–25 minutes** hands-on (capstone)

## Scenario

You are the **ML Evaluation Lead** at a legal-tech company. Your team ships an AI assistant called **LegalBot** that:

1. **Extracts structured fields** from contracts (party names, dates, amounts, clauses)
2. **Answers questions** about a legal knowledge base using RAG (e.g., "What are the termination conditions in Contract #42?")
3. **Generates executive summaries** of contract reviews for stakeholders

A new model version (**v2.1**) is ready for evaluation against the current production version (**v2.0**). Leadership wants a go/no-go decision backed by data. Your job: build the evaluation pipeline, run it, and make the call.

> 🎯 **Why This Matters:** In production AI, shipping without a rigorous evaluation pipeline is like deploying software without tests. One bad release in a legal context could cause real harm to real people.

---

## Task 1: Environment Setup and Dependencies 🛠️

### Goal
Install and import the required packages for the evaluation pipeline.

### Why Start Here?
Every evaluation pipeline depends on reliable data manipulation and analysis tools. We use **pandas** for tabular data (evaluation results live in tables) and **numpy** for numerical operations (computing averages, thresholds, and deltas). Getting the environment right first means no surprises later.

### Steps
Run the cells below to install dependencies and import libraries.

In [ ]:
%%capture
# ============================================================
# Install required packages for the capstone evaluation pipeline
# ============================================================
# pandas — DataFrames for tabular evaluation results. Every evaluation
#           metric, test case, and comparison lives in a DataFrame.
# numpy  — Numerical operations: computing means, deltas, thresholds,
#           and statistical comparisons between model versions.
#
# NOTE: This capstone lab intentionally uses ONLY basic data tools
# (no LLM SDKs, no Langfuse). The focus is on evaluation pipeline
# DESIGN and DECISION-MAKING, not API integration. In a real system,
# the data would come from Langfuse traces and automated evaluation runs.
!pip install pandas numpy

In [ ]:
# ============================================================
# Import libraries for the evaluation pipeline
# ============================================================

import pandas as pd               # Tabular data — evaluation results live in DataFrames.
                                   # Every metric, test case, and version comparison is
                                   # organized as rows and columns for easy aggregation.

import numpy as np                # Numerical ops — computing averages, deltas between
                                   # model versions, and threshold comparisons.

import json                       # For structured output. Scorecards and pipeline reports
                                   # are JSON-serializable dicts that can be logged, stored,
                                   # and passed between pipeline stages.

from datetime import datetime     # Timestamps for pipeline reports — essential for
                                   # traceability ("when was this evaluation run?").

from collections import defaultdict  # Handy for grouping results by component, metric,
                                      # or severity without pre-initializing every key.

print("✅ All imports successful.")

### Checkpoint ✅
You should see `✅ All imports successful.` with no errors. If you see a `ModuleNotFoundError`, re-run the install cell and restart the kernel.

Now that our tools are ready, let's load the evaluation data we'll be working with throughout this lab. ⬇️

---

## Task 2: Load Multi-Component System Evaluation Data 📊

### Goal
Load synthetic evaluation results for two versions of LegalBot (v2.0 and v2.1). The data covers all three components: structured extraction, RAG Q&A, and text generation.

### Why This Data Structure?
In a real production system, you evaluate **multiple components independently** because each has different failure modes and different metrics that matter. A single "model score" would hide critical details — like a safety regression in one component being masked by improvements in another.

> ⚠️ **Common Mistake:** Averaging all metrics across all components into one number. This hides regressions and makes it impossible to pinpoint where problems are.

### How to Read This Data
Each evaluation record represents one test case evaluated on both model versions. The dataset includes:
- **Component**: Which subsystem was tested (`structured`, `rag`, `generation`)
- **Test case ID and description**: What specific scenario was tested
- **v2.0 and v2.1 scores**: Performance on each version (0.0–1.0 scale)
- **Metric name**: What was measured (e.g., `field_accuracy`, `faithfulness`, `safety`)

### Steps
Run the cell below to create the evaluation dataset.

In [ ]:
# ============================================================
# Synthetic Evaluation Data: LegalBot v2.0 vs v2.1
# ============================================================
# This dataset represents evaluation results for a multi-component
# legal AI assistant. Each record is a test case with scores from
# BOTH model versions, enabling direct comparison.
#
# IN A REAL SYSTEM, this data would come from running your automated
# test suite against both model versions — the inputs and expected
# outputs are the same, only the model changes.
#
# DATASET STRUCTURE:
#   component   — Which subsystem: structured, rag, or generation
#   metric      — What was measured (e.g., field_accuracy, faithfulness)
#   test_id     — Unique identifier for this test case (S-001, R-001, G-001)
#   description — Human-readable description of what's being tested
#   v2_0_score  — Performance on the CURRENT production model (0.0–1.0)
#   v2_1_score  — Performance on the NEW candidate model (0.0–1.0)
#
# KEY PATTERN TO DISCOVER:
# The data contains INTENTIONAL REGRESSIONS (marked with ← comments).
# v2.1 generally improves on standard/simple cases but regresses on
# complex, noisy, or edge-case inputs. This is a CLASSIC pattern
# when models are fine-tuned — they optimize for common cases at
# the expense of the tail distribution.
#
# YOUR JOB: Find these regressions and decide whether they're
# acceptable for production deployment.

eval_data = [
    # ══════════════════════════════════════════════════════════
    # COMPONENT 1: Structured Extraction
    # ══════════════════════════════════════════════════════════
    # What it does: Pulls structured fields (names, dates, amounts)
    # from legal contracts. Think OCR + NER + schema formatting.
    # Why it matters: Downstream systems consume these fields as JSON.
    # If extraction is wrong, downstream decisions are wrong too.

    # ── Metric: field_accuracy ──
    # Measures: Exact match of extracted fields against ground truth.
    # "Did we get the party name / date / amount right?"
    # Scale: 0.0 (all fields wrong) to 1.0 (all fields correct)
    {"component": "structured", "metric": "field_accuracy", "test_id": "S-001",
     "description": "Extract party names from standard contract",
     "v2_0_score": 0.95, "v2_1_score": 0.97},  # ✅ improved — standard case

    {"component": "structured", "metric": "field_accuracy", "test_id": "S-002",
     "description": "Extract dates from amendment document",
     "v2_0_score": 0.90, "v2_1_score": 0.93},  # ✅ improved — date extraction

    {"component": "structured", "metric": "field_accuracy", "test_id": "S-003",
     "description": "Extract monetary amounts with currency",
     "v2_0_score": 0.88, "v2_1_score": 0.91},  # ✅ improved — currency handling

    {"component": "structured", "metric": "field_accuracy", "test_id": "S-004",
     "description": "Extract clause types from multi-party agreement",
     "v2_0_score": 0.82, "v2_1_score": 0.85},  # ✅ improved — complex doc

    {"component": "structured", "metric": "field_accuracy", "test_id": "S-005",
     "description": "Extract fields from poorly scanned document (OCR noise)",
     "v2_0_score": 0.71, "v2_1_score": 0.68},  # ← REGRESSION on noisy input!
     # WHY THIS MATTERS: OCR-degraded documents are common in legal practice.
     # If v2.1 handles clean docs better but noisy docs worse, that's a
     # real-world problem — you can't always guarantee scan quality.

    # ── Metric: schema_compliance ──
    # Measures: Does the output match the expected JSON schema?
    # This checks STRUCTURAL correctness — even if values are wrong,
    # is the output in the right format for downstream consumers?
    # A schema violation = broken pipeline downstream.
    {"component": "structured", "metric": "schema_compliance", "test_id": "S-006",
     "description": "Schema compliance on standard contracts",
     "v2_0_score": 0.98, "v2_1_score": 0.99},  # near-perfect, slight improvement

    {"component": "structured", "metric": "schema_compliance", "test_id": "S-007",
     "description": "Schema compliance on edge-case formats",
     "v2_0_score": 0.91, "v2_1_score": 0.89},  # ← REGRESSION on edge cases!
     # This is concerning: non-standard document formats trigger schema
     # violations more often in v2.1. This could break production pipelines.

    {"component": "structured", "metric": "schema_compliance", "test_id": "S-008",
     "description": "Schema compliance with optional fields present",
     "v2_0_score": 0.94, "v2_1_score": 0.96},  # ✅ improved — optional field handling

    # ══════════════════════════════════════════════════════════
    # COMPONENT 2: RAG Q&A
    # ══════════════════════════════════════════════════════════
    # What it does: Retrieves relevant legal documents and answers
    # questions based on them. A "search + answer" pipeline.
    # Why it matters: Wrong answers in legal context = real-world harm.
    # Hallucinated legal advice could lead to financial/legal consequences.

    # ── Metric: retrieval_recall ──
    # Measures: Did we find the RIGHT documents?
    # This is measured BEFORE the LLM generates an answer.
    # If retrieval misses relevant docs, the LLM can't use them.
    # Scale: 0.0 (no relevant docs found) to 1.0 (all relevant docs found)
    {"component": "rag", "metric": "retrieval_recall", "test_id": "R-001",
     "description": "Retrieve termination clauses for Contract #42",
     "v2_0_score": 0.85, "v2_1_score": 0.88},  # ✅ improved

    {"component": "rag", "metric": "retrieval_recall", "test_id": "R-002",
     "description": "Retrieve indemnification details across related contracts",
     "v2_0_score": 0.78, "v2_1_score": 0.81},  # ✅ improved

    {"component": "rag", "metric": "retrieval_recall", "test_id": "R-003",
     "description": "Retrieve renewal terms from expired contracts",
     "v2_0_score": 0.80, "v2_1_score": 0.75},  # ← REGRESSION on temporal queries!
     # Temporal reasoning ("expired contracts") is notoriously hard for
     # retrieval systems. New embeddings may not encode time as well.

    {"component": "rag", "metric": "retrieval_recall", "test_id": "R-004",
     "description": "Retrieve confidentiality obligations for Party B",
     "v2_0_score": 0.90, "v2_1_score": 0.92},  # ✅ improved

    # ── Metric: faithfulness ──
    # Measures: Is the answer GROUNDED in the retrieved documents?
    # A "faithful" answer only states things supported by the documents.
    # Unfaithful = hallucinated legal advice = VERY DANGEROUS.
    # This is the single most critical metric for a legal RAG system.
    {"component": "rag", "metric": "faithfulness", "test_id": "R-005",
     "description": "Faithfulness on single-document questions",
     "v2_0_score": 0.92, "v2_1_score": 0.94},  # ✅ improved on simple case

    {"component": "rag", "metric": "faithfulness", "test_id": "R-006",
     "description": "Faithfulness on multi-document synthesis",
     "v2_0_score": 0.84, "v2_1_score": 0.79},  # ← REGRESSION on complex synthesis!
     # THIS IS THE MOST DANGEROUS REGRESSION IN THE DATASET.
     # When the model needs to synthesize info across multiple documents,
     # it's MORE likely to hallucinate in v2.1. In a legal context,
     # a hallucinated cross-reference could lead to wrong legal advice.

    {"component": "rag", "metric": "faithfulness", "test_id": "R-007",
     "description": "Faithfulness when no relevant document retrieved",
     "v2_0_score": 0.70, "v2_1_score": 0.73},  # ✅ improved — better at saying "I don't know"

    {"component": "rag", "metric": "faithfulness", "test_id": "R-008",
     "description": "Faithfulness on adversarial questions",
     "v2_0_score": 0.65, "v2_1_score": 0.68},  # ✅ improved — more robust

    # ── Metric: answer_correctness ──
    # Measures: Is the final answer factually correct?
    # Different from faithfulness: an answer can be faithful to the
    # retrieved docs but still wrong (if the wrong docs were retrieved).
    # This is the end-to-end correctness check.
    {"component": "rag", "metric": "answer_correctness", "test_id": "R-009",
     "description": "Factual correctness on standard queries",
     "v2_0_score": 0.88, "v2_1_score": 0.90},  # ✅ improved

    {"component": "rag", "metric": "answer_correctness", "test_id": "R-010",
     "description": "Factual correctness on complex multi-hop queries",
     "v2_0_score": 0.75, "v2_1_score": 0.71},  # ← REGRESSION on complex queries!
     # Multi-hop queries require chaining information across documents.
     # Same pattern as R-006: v2.1 struggles with COMPLEX synthesis.

    {"component": "rag", "metric": "answer_correctness", "test_id": "R-011",
     "description": "Factual correctness on numerical/date questions",
     "v2_0_score": 0.82, "v2_1_score": 0.84},  # ✅ improved

    # ══════════════════════════════════════════════════════════
    # COMPONENT 3: Generation
    # ══════════════════════════════════════════════════════════
    # What it does: Generates executive summaries of contract reviews.
    # Takes structured data + Q&A results → human-readable summaries.
    # Why it matters: These summaries go to executives and lawyers.
    # Quality, completeness, and safety all matter.

    # ── Metric: coherence ──
    # Measures: Is the summary well-structured and readable?
    # A coherent summary has logical flow, clear sections, and
    # no self-contradicting statements.
    {"component": "generation", "metric": "coherence", "test_id": "G-001",
     "description": "Executive summary coherence — simple contract",
     "v2_0_score": 0.88, "v2_1_score": 0.92},  # ✅ improved

    {"component": "generation", "metric": "coherence", "test_id": "G-002",
     "description": "Executive summary coherence — complex multi-party deal",
     "v2_0_score": 0.79, "v2_1_score": 0.85},  # ✅ big improvement

    {"component": "generation", "metric": "coherence", "test_id": "G-003",
     "description": "Executive summary coherence — amendment chain",
     "v2_0_score": 0.82, "v2_1_score": 0.87},  # ✅ improved

    # ── Metric: completeness ──
    # Measures: Does the summary cover ALL key points from the source?
    # An incomplete summary that misses a critical clause is dangerous
    # in legal contexts — decision-makers might not know what's missing.
    {"component": "generation", "metric": "completeness", "test_id": "G-004",
     "description": "Summary completeness — standard contract review",
     "v2_0_score": 0.85, "v2_1_score": 0.88},  # ✅ improved

    {"component": "generation", "metric": "completeness", "test_id": "G-005",
     "description": "Summary completeness — high-risk contract flagging",
     "v2_0_score": 0.80, "v2_1_score": 0.83},  # ✅ improved

    {"component": "generation", "metric": "completeness", "test_id": "G-006",
     "description": "Summary completeness — cross-reference multiple docs",
     "v2_0_score": 0.77, "v2_1_score": 0.72},  # ← REGRESSION on cross-reference!
     # PATTERN: v2.1 writes "better" summaries (more coherent) but
     # MISSES information when cross-referencing multiple documents.
     # Same theme: single-doc = better, multi-doc = worse.

    # ── Metric: safety ──
    # Measures: No hallucinated legal advice, proper disclaimers included.
    # Safety is CRITICAL in legal AI — a wrong legal interpretation
    # could lead to multimillion-dollar consequences. Even a small
    # regression (e.g., 0.02 drop) demands serious investigation.
    {"component": "generation", "metric": "safety", "test_id": "G-007",
     "description": "Safety — no unauthorized legal interpretations",
     "v2_0_score": 0.95, "v2_1_score": 0.93},  # ← SAFETY REGRESSION!
     # ANY safety regression in a legal system is a red flag.
     # The model is now 2% more likely to generate unauthorized
     # legal interpretations. In legal, 2% matters.

    {"component": "generation", "metric": "safety", "test_id": "G-008",
     "description": "Safety — includes proper disclaimers",
     "v2_0_score": 0.92, "v2_1_score": 0.94},  # ✅ improved

    {"component": "generation", "metric": "safety", "test_id": "G-009",
     "description": "Safety — handles ambiguous legal questions appropriately",
     "v2_0_score": 0.88, "v2_1_score": 0.85},  # ← REGRESSION on ambiguous inputs!
     # The model handles clear-cut questions better but struggles MORE
     # with ambiguous ones. In law, ambiguity is everywhere.

    # ── Metric: tone ──
    # Measures: Professional and appropriate for legal context.
    # An overly casual or sensational tone undermines credibility.
    {"component": "generation", "metric": "tone", "test_id": "G-010",
     "description": "Professional tone in executive summaries",
     "v2_0_score": 0.90, "v2_1_score": 0.93},  # ✅ improved

    {"component": "generation", "metric": "tone", "test_id": "G-011",
     "description": "Tone consistency across summary sections",
     "v2_0_score": 0.87, "v2_1_score": 0.91},  # ✅ improved
]

# ============================================================
# Convert to DataFrame and add computed columns
# ============================================================
df = pd.DataFrame(eval_data)

# DELTA = how much each test case changed from v2.0 → v2.1
# Positive delta = improvement, negative delta = regression
df["delta"] = df["v2_1_score"] - df["v2_0_score"]

# Boolean flags for quick filtering — used in aggregations later
df["improved"] = df["delta"] > 0    # Did this test case get better?
df["regressed"] = df["delta"] < 0   # Did this test case get worse?

# Print a high-level summary so we can see the big picture BEFORE
# diving into details. This is the "30-second overview" that tells
# us whether v2.1 is broadly better or broadly worse.
print(f"📊 Loaded {len(df)} evaluation test cases")
print(f"   Components: {df['component'].nunique()} — {', '.join(df['component'].unique())}")
print(f"   Metrics: {df['metric'].nunique()} — {', '.join(df['metric'].unique())}")
print(f"   Improved in v2.1: {df['improved'].sum()}")
print(f"   Regressed in v2.1: {df['regressed'].sum()}")
print(f"   Unchanged: {(df['delta'] == 0).sum()}")

In [ ]:
# ============================================================
# Summary Aggregation: Component × Metric Overview
# ============================================================
# WHY AGGREGATE? Looking at 31 individual test cases is overwhelming.
# Grouping by component and metric reveals the big-picture trends.
#
# CAVEAT: Averages can HIDE regressions!
# If 4 out of 5 test cases improved by +0.03 and one regressed by -0.05,
# the average delta is +0.01 (looks fine!) but there's a real problem.
# That's why we also count regressions alongside the averages.

summary = df.groupby(["component", "metric"]).agg(
    num_tests=("test_id", "count"),         # How many test cases per component+metric
    v2_0_mean=("v2_0_score", "mean"),       # Average v2.0 score (our baseline)
    v2_1_mean=("v2_1_score", "mean"),       # Average v2.1 score (candidate)
    avg_delta=("delta", "mean"),            # Average improvement (+ good, - bad)
    regressions=("regressed", "sum"),       # Count of individual test cases that got WORSE
).round(3)

# Display the summary — scan for any rows where regressions > 0
# even if avg_delta is positive. Those are the hidden problem areas.
print("=" * 70)
print("EVALUATION SUMMARY: v2.0 → v2.1")
print("=" * 70)
print(summary.to_string())
print("\n💡 Tip: Look for rows where avg_delta > 0 but regressions > 0.")
print("   Those are the metrics where 'looks good on average' hides problems.")

### Checkpoint ✅
You should see:
- **30 evaluation test cases** loaded (22 improved, 8 regressed)
- **3 components**: structured, rag, generation
- **9 metrics** across all components
- A mix of improvements and regressions in the summary table

Take a moment to scan the summary table. Which components improved overall? Which have regressions?

### 🔍 What Just Happened?

You just loaded the **evaluation dataset** that will drive every decision in this lab. Notice the key pattern: **most metrics improved** from v2.0 to v2.1, but **several specific test cases regressed**. This is exactly the kind of ambiguity you'll face in real production evaluations — overall improvement with pockets of regression. The rest of this lab is about building a systematic way to handle that ambiguity.

Now let's think about *how* to organize the evaluation before writing any pipeline code. ⬇️

---

## Task 3: 🔍 Think First — Evaluation Stages

### Goal
Before building the pipeline, reason about **what evaluation should happen at each stage** of the deployment process.

### Why Think Before Coding?
Jumping straight to code is tempting, but **the design of your evaluation stages determines what problems you catch and when**. A poorly designed pipeline either misses critical issues or wastes time running expensive checks on obviously broken builds.

> 💡 **Key Insight:** The evaluation pipeline is like a funnel — fast, cheap checks first (to reject bad builds quickly), expensive thorough checks later (only on promising candidates).

### Context
A typical deployment pipeline has three stages:

```
┌──────────────────────────────────────────────────────────────┐
│                  DEPLOYMENT PIPELINE                         │
│                                                              │
│   Stage 1: CI           Stage 2: Pre-Prod    Stage 3: Prod  │
│   ┌────────────┐        ┌──────────────┐     ┌────────────┐ │
│   │ Every      │───────▶│ Before       │────▶│ Continuous │ │
│   │ commit     │  pass  │ release      │ pass│ monitoring │ │
│   │ Seconds-   │        │ Minutes-     │     │ Always on  │ │
│   │ minutes    │        │ hours        │     │            │ │
│   └────────────┘        └──────────────┘     └────────────┘ │
│    Fast & cheap          Thorough             Real-world     │
│    Golden test set       Full test suite      Live traffic   │
│    Catch breakages       Catch regressions    Catch drift    │
└──────────────────────────────────────────────────────────────┘
```

1. **CI (Continuous Integration)** — runs on every code change; must be fast (seconds to minutes)
2. **Pre-production** — runs before a release candidate goes live; can be thorough (minutes to hours)
3. **Production** — runs continuously on live traffic; monitors for degradation

### 🔍 Your Task

In the cell below, write your answers to these questions:

1. **Which of the 9 metrics in our dataset should be checked at CI?** Why those and not others?
2. **What additional evaluation should happen at pre-production that doesn't happen at CI?**
3. **What would you monitor in production that you can't fully test pre-production?**
4. **For each stage, what's the acceptable latency budget for evaluation?**

Spend 2–3 minutes thinking before writing. There is no single correct answer — but some choices are clearly better than others.

**Hint:** Think about what's fast to compute vs. what requires expensive LLM calls. Think about what catches critical issues vs. what catches subtle quality shifts.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🔍 THINK FIRST: Write your answers here                       ║
# ╚══════════════════════════════════════════════════════════════════╝
# Take a moment to think about each question BEFORE writing.
# There's no auto-grading — this is about building your intuition
# for what belongs at each stage of a deployment pipeline.
#
# Good evaluation pipeline design is like a medical triage system:
#   - Emergency room (CI): catch life-threatening issues instantly
#   - Specialist exam (pre-prod): thorough investigation of everything
#   - Ongoing monitoring (production): watch for new symptoms over time

# 1. Which metrics should be checked at CI? Why?
# ────────────────────────────────────────────────
# Think about speed vs. thoroughness tradeoff:
#   - CI runs on EVERY commit — developers wait for it
#   - Only include metrics computable on a small "golden set" (< 50 examples)
#   - Focus on "if this breaks, nothing else matters" metrics
#   - Avoid metrics requiring expensive LLM calls (faithfulness, coherence)
ci_metrics_reasoning = """
YOUR ANSWER HERE
"""

# 2. What additional evaluation at pre-production?
# ────────────────────────────────────────────────
# Pre-prod has more time budget (30-60 min instead of 2 min):
#   - Full test suite (hundreds/thousands of examples, not just golden set)
#   - LLM-as-judge evaluations (faithfulness, coherence — expensive but thorough)
#   - Cross-component tests (does RAG + generation work end-to-end?)
#   - Regression analysis (compare every metric against the previous version)
pre_prod_reasoning = """
YOUR ANSWER HERE
"""

# 3. What to monitor in production?
# ────────────────────────────────────────────────
# Production monitoring catches things test suites CAN'T:
#   - Distribution shift: real user queries ≠ test queries
#   - Edge cases you didn't think to test
#   - Gradual degradation over weeks (model drift, data staleness)
#   - Latency/cost under real load patterns
#   - User satisfaction signals (thumbs up/down, complaints)
production_reasoning = """
YOUR ANSWER HERE
"""

# 4. Acceptable latency budget per stage?
# ────────────────────────────────────────────────
# Think about who's waiting and what they're waiting FOR:
#   - CI: developer waiting to merge → seconds to low minutes
#   - Pre-prod: release process → minutes to an hour is fine
#   - Production: nobody waits, but alerts must fire quickly
latency_reasoning = """
YOUR ANSWER HERE
"""

print("✍️  Answers recorded. Discuss with the class before moving on.")

### Checkpoint ✅
You should have written thoughtful answers for all four questions. Your instructor will facilitate a brief discussion before moving on.

> 💡 **Key insight:** CI should catch obvious breakages fast (schema compliance, field accuracy on golden test cases). Pre-production should run the full suite including expensive faithfulness and safety checks. Production should monitor real-world distributions that test suites can't anticipate.

### 🔍 What Just Happened?

You just practiced the most important skill in evaluation pipeline design: **deciding what to check when**. This "evaluation triage" determines the efficiency and effectiveness of your entire pipeline. Now let's implement those stage definitions in code. ⬇️

---

## Task 4: Design Evaluation Stages with Metrics ⚙️

### Goal
Now implement the evaluation stage definitions. You will assign each metric to one or more stages and configure appropriate thresholds.

### Why Define Stages Explicitly?
Writing down the stage definitions in code makes the pipeline **reproducible and auditable**. When leadership asks "what checks did you run before deploying v2.1?", you can point to this exact configuration — no ambiguity.

> ⚠️ **Common Mistake:** Running all checks at every stage wastes time and resources. CI should be fast (4 checks), not comprehensive (9+ checks). Save the thorough evaluation for pre-production.

### Steps
Run the cell below to create the stage configuration, then review the assignments.

In [ ]:
# ============================================================
# Evaluation Stage Definitions
# ============================================================
# This is the "evaluation contract" — the explicit, codified rules
# your pipeline enforces at each stage of the deployment process.
#
# DESIGN PHILOSOPHY:
#   1. CI = fast & cheap → catch breakages early (seconds, not minutes)
#   2. Pre-prod = slow & thorough → catch regressions before release
#   3. Production = always-on → catch drift and real-world issues
#
# Each stage defines:
#   - which metrics to check (scope)
#   - what thresholds to apply (quality bar)
#   - what action to take on failure (consequence)
#   - a rationale (WHY this threshold, for audit trail)
#
# THRESHOLD DESIGN RULES:
#   - CI thresholds are LENIENT → minimize false positives that block devs
#   - Pre-prod thresholds are STRICT → this is the last gate before users
#   - Safety thresholds are ALWAYS strict → even at CI
#
# ACTION TYPES:
#   - "BLOCK merge"/"BLOCK release" → hard gate, cannot proceed
#   - "WARN — review required" → soft gate, needs human review

evaluation_stages = {
    # ── STAGE 1: CI / Pre-Commit ──────────────────────────────────
    # PURPOSE: Fast smoke test on every code change
    # BUDGET: Must complete in < 2 minutes (developers are waiting!)
    # STRATEGY: Only 4 checks — the absolute must-not-break metrics
    # WHY ONLY 4? CI runs 50+ times/day. Keeping it fast means devs
    # actually wait for it instead of merging without checking.
    "ci": {
        "name": "CI / Pre-Commit",
        "description": "Fast checks on every commit. Must complete in < 2 minutes.",
        "checks": [
            {
                "component": "structured",
                "metric": "schema_compliance",
                "threshold": 0.90,
                "action_on_fail": "BLOCK merge",
                # WHY BLOCK: Schema violations break downstream consumers IMMEDIATELY.
                # A broken schema = broken pipeline for every team that depends on us.
                "rationale": "Schema violations break downstream consumers immediately"
            },
            {
                "component": "structured",
                "metric": "field_accuracy",
                "threshold": 0.80,
                "action_on_fail": "BLOCK merge",
                # WHY 80%: This is run on a small golden set (~20 examples).
                # Below 80% on golden examples = something is fundamentally broken.
                "rationale": "Extraction accuracy below 80% indicates a fundamental issue"
            },
            {
                "component": "rag",
                "metric": "answer_correctness",
                "threshold": 0.75,
                "action_on_fail": "BLOCK merge",
                # WHY 75%: On a golden set, 75% is a low bar intentionally.
                # We'll check more thoroughly at pre-prod.
                "rationale": "Basic factual correctness — fast to check on golden set"
            },
            {
                "component": "generation",
                "metric": "safety",
                "threshold": 0.90,
                "action_on_fail": "BLOCK merge",
                # WHY 90% EVEN AT CI: Safety regressions are the one thing
                # you should catch as early as possible. The cost of a
                # safety incident far exceeds the cost of a blocked merge.
                "rationale": "Safety regressions must never reach production"
            },
        ]
    },

    # ── STAGE 2: Pre-Production ───────────────────────────────────
    # PURPOSE: Comprehensive evaluation before a release goes live
    # BUDGET: 30–60 minutes — no developer is waiting for this
    # STRATEGY: Check ALL metrics with stricter thresholds
    # WHY 9 CHECKS? This is the last automated gate before users.
    # It's worth running expensive LLM-based evaluators here.
    "pre_production": {
        "name": "Pre-Production",
        "description": "Comprehensive evaluation before release. Can take 30–60 minutes.",
        "checks": [
            {
                "component": "structured",
                "metric": "field_accuracy",
                "threshold": 0.85,
                "action_on_fail": "BLOCK release",
                # WHY 85% (vs 80% at CI): Full test suite, not just golden set.
                # Higher bar because we have more test cases and more time.
                "rationale": "Higher bar than CI — full test suite"
            },
            {
                "component": "structured",
                "metric": "schema_compliance",
                "threshold": 0.95,
                "action_on_fail": "BLOCK release",
                # WHY 95%: Near-perfect schema compliance is essential.
                # Even 5% schema violations means 1 in 20 outputs break.
                "rationale": "Near-perfect schema compliance required for release"
            },
            {
                "component": "rag",
                "metric": "retrieval_recall",
                "threshold": 0.80,
                "action_on_fail": "BLOCK release",
                # WHY BLOCK: Retrieval is the FOUNDATION of RAG.
                # If recall drops, the LLM can't use docs it never retrieved.
                # Everything downstream of retrieval gets worse.
                "rationale": "Retrieval is the foundation — if recall drops, everything downstream suffers"
            },
            {
                "component": "rag",
                "metric": "faithfulness",
                "threshold": 0.80,
                "action_on_fail": "BLOCK release",
                # WHY BLOCK on faithfulness: In a legal context, an unfaithful
                # answer = hallucinated legal advice = potential liability.
                # This is a safety-adjacent metric for legal applications.
                "rationale": "Unfaithful answers in a legal context are dangerous"
            },
            {
                "component": "rag",
                "metric": "answer_correctness",
                "threshold": 0.80,
                "action_on_fail": "BLOCK release",
                # WHY 80%: Correctness is the end-to-end measure.
                # If faithfulness is high but correctness is low,
                # it means we're retrieving the wrong documents.
                "rationale": "Must maintain factual accuracy across full test suite"
            },
            {
                "component": "generation",
                "metric": "coherence",
                "threshold": 0.80,
                "action_on_fail": "WARN — review required",
                # WHY WARN (not BLOCK): A less-coherent summary is embarrassing
                # but not dangerous. Worth investigating but shouldn't block release.
                "rationale": "Coherence regression is quality concern but not safety-critical"
            },
            {
                "component": "generation",
                "metric": "completeness",
                "threshold": 0.78,
                "action_on_fail": "WARN — review required",
                # WHY 78%: Completeness is important but hard to get perfect.
                # Missing one minor point in a summary ≠ a safety issue.
                "rationale": "Incomplete summaries reduce user trust"
            },
            {
                "component": "generation",
                "metric": "safety",
                "threshold": 0.92,
                "action_on_fail": "BLOCK release",
                # WHY 92% (stricter than CI's 90%): Pre-prod is the last gate.
                # We tighten the safety threshold because there's no more
                # automated checks after this — just production monitoring.
                "rationale": "Stricter safety threshold at pre-production"
            },
            {
                "component": "generation",
                "metric": "tone",
                "threshold": 0.85,
                "action_on_fail": "WARN — review required",
                # WHY WARN: Unprofessional tone is a brand risk, not a safety
                # risk. Worth reviewing but shouldn't hold up a release.
                "rationale": "Unprofessional tone is a brand risk"
            },
        ]
    },

    # ── STAGE 3: Production Monitoring ────────────────────────────
    # PURPOSE: Continuously watch live traffic for degradation
    # BUDGET: Always on — alerts trigger within minutes
    # STRATEGY: Monitor the most DANGEROUS failure modes only.
    # Production monitoring is NOT about comprehensiveness — it's about
    # catching the failure modes that cause the most harm, fastest.
    #
    # KEY DIFFERENCE from CI/pre-prod:
    #   - Uses time WINDOWS (rolling averages) instead of single test runs
    #   - Has AUTOMATED ACTIONS (circuit breakers, auto-disable)
    #   - Monitors things you CAN'T test pre-prod (latency under real load)
    "production": {
        "name": "Production Monitoring",
        "description": "Continuous monitoring on live traffic. Alerts on degradation.",
        "monitors": [
            {
                "component": "structured",
                "metric": "schema_compliance",
                "alert_threshold": 0.90,
                "window": "1 hour rolling",
                # WHY 1 HOUR: Schema violations are easy to detect and
                # usually indicate a systemic problem, not random noise.
                "action": "Page on-call if below threshold"
            },
            {
                "component": "rag",
                "metric": "faithfulness",
                "alert_threshold": 0.75,
                "window": "1 hour rolling",
                # WHY AUTO-DISABLE: If the RAG system is hallucinating,
                # the fallback is to disable it entirely and show users
                # a "search not available" message. Better than bad answers.
                "action": "Page on-call + auto-disable RAG fallback"
            },
            {
                "component": "generation",
                "metric": "safety",
                "alert_threshold": 0.88,
                "window": "15 minute rolling",
                # WHY 15 MIN (tighter window): Safety issues can cause
                # harm in minutes. A 1-hour window means 45 minutes
                # of potentially harmful outputs before we react.
                # Circuit breaker = automatic shutdown of generation.
                "action": "Immediate page + circuit breaker"
            },
            {
                "component": "all",
                "metric": "latency_p95",
                "alert_threshold": "5000ms",
                "window": "5 minute rolling",
                # WHY P95 LATENCY: If 5% of requests take >5 seconds,
                # users will abandon the tool. P95 catches tail latency
                # that P50 (median) would miss.
                "action": "Alert engineering team"
            },
        ]
    }
}

# ── Display a summary of all stages for review ──
for stage_key, stage in evaluation_stages.items():
    check_key = "checks" if "checks" in stage else "monitors"
    items = stage[check_key]
    print(f"\n{'='*60}")
    print(f"📋 {stage['name']}")
    print(f"   {stage['description']}")
    print(f"   Items: {len(items)}")
    for item in items:
        label = item.get("metric", "unknown")
        comp = item.get("component", "all")
        thresh = item.get("threshold", item.get("alert_threshold", "N/A"))
        print(f"   • [{comp}] {label} — threshold: {thresh}")

### Checkpoint ✅
You should see three evaluation stages printed:
- **CI / Pre-Commit**: 4 checks (fast, critical)
- **Pre-Production**: 9 checks (comprehensive)
- **Production Monitoring**: 4 monitors (continuous)

Review the stage assignments. Do they make sense? Is there anything you'd add or remove?

### 🔍 What Just Happened?

You defined the **evaluation contract** — the explicit rules your pipeline will enforce. Notice the design principles:
- CI has **few checks with lenient thresholds** (fast feedback)
- Pre-production has **many checks with strict thresholds** (thorough vetting)
- Production monitors the **most dangerous failure modes** (safety, faithfulness) with narrow time windows

Next up: you'll implement the **gating logic** that turns these check results into PASS/WARN/FAIL decisions. ⬇️

---

## Task 5: 🧠 Your Turn — Define Gating Criteria

### Goal
Now define the **gating logic** — the rules that decide whether a model version passes or fails each stage. This is where evaluation becomes decision-making.

### Why Gating Matters
Without explicit gating criteria, evaluation results are just numbers on a page. **Gating logic is the bridge between metrics and decisions.** It answers: "Given these results, should we proceed to the next stage — yes, no, or maybe?"

```
  Check Results             Gating Logic               Decision
┌──────────────┐    ┌─────────────────────┐    ┌──────────────────┐
│ safety: 0.93 │───▶│ Any BLOCK fail?     │───▶│ ❌ FAIL          │
│ schema: 0.99 │    │ How many WARNs?     │    │ ⚠️ WARN          │
│ accuracy:0.87│    │ All passed?         │    │ ✅ PASS          │
└──────────────┘    └─────────────────────┘    └──────────────────┘
```

### 🧠 Your Task

Complete the `define_gating_criteria()` function below. For each stage, you must decide:

1. **What must pass for the stage to be "PASS"?**
2. **What combination triggers "WARN" vs "FAIL"?**
3. **How do you handle the difference between BLOCK and WARN checks?**

The function should return a dictionary with a `status` ("PASS", "WARN", or "FAIL") and a list of `reasons`.

> 💡 **Key Insight:** A single BLOCK failure should fail the stage — these are your hard safety/quality lines. WARN failures need judgment: one WARN might be acceptable, but three WARNs together could indicate a systemic issue.

**Hint:** Think about it this way — a single BLOCK failure should fail the stage. Multiple WARNs might accumulate into a failure. But what about one WARN and everything else passing?

In [ ]:
# ============================================================
# Stage Evaluation Runner + Gating Criteria
# ============================================================
# TWO functions here:
#   1. run_stage_evaluation() — runs the checks; produces raw results
#   2. define_gating_criteria() — YOUR TURN; interprets results into a decision
#
# Why separate the two? This is a classic "data vs. decision" separation:
#   - run_stage_evaluation produces FACTS ("safety scored 0.93 vs threshold 0.92")
#   - define_gating_criteria applies JUDGMENT ("is this a PASS, WARN, or FAIL?")
# Keeping them separate means you can change the decision logic without
# re-running the evaluations, and vice versa.

def run_stage_evaluation(stage_config, eval_df):
    """
    Run evaluation checks for a given pipeline stage against the data.

    For each check defined in the stage config, this function:
    1. Filters the eval DataFrame for matching component + metric
    2. Computes the mean v2.1 score across all matching test cases
    3. Compares the mean against the threshold
    4. Records the result (pass/fail) along with metadata

    Args:
        stage_config: dict with "checks" list (from evaluation_stages)
        eval_df: DataFrame with columns [component, metric, v2_1_score, ...]

    Returns:
        List of dicts, each containing:
          - component, metric: what was checked
          - mean_score: actual v2.1 performance (averaged across test cases)
          - threshold: minimum acceptable score
          - passed: bool (did it meet the threshold?)
          - action: what to do if it fails ("BLOCK" or "WARN")
          - detail: human-readable summary string
    """
    results = []
    checks = stage_config.get("checks", [])

    for check in checks:
        comp = check["component"]
        metric = check["metric"]
        threshold = check["threshold"]
        action = check["action_on_fail"]

        # Filter the DataFrame to get ONLY the v2.1 scores for this
        # specific component + metric combination
        mask = (eval_df["component"] == comp) & (eval_df["metric"] == metric)
        scores = eval_df.loc[mask, "v2_1_score"]

        # Edge case: no test cases found for this component+metric
        # This could happen if the test suite is incomplete or if there's
        # a typo in the stage config. Treat it as a failure to be safe.
        if len(scores) == 0:
            results.append({
                "component": comp,
                "metric": metric,
                "mean_score": None,
                "threshold": threshold,
                "passed": False,
                "action": action,
                "detail": "No test cases found — check config for typos"
            })
            continue

        # Compare the AVERAGE score against the threshold.
        # NOTE: Using the mean is a simplification. In production, you might
        # also check the MIN score or a percentile (e.g., "no test case below 0.70").
        mean_score = scores.mean()
        passed = mean_score >= threshold

        results.append({
            "component": comp,
            "metric": metric,
            "mean_score": round(mean_score, 3),
            "threshold": threshold,
            "passed": passed,
            "action": action,
            "detail": f"{'✅ PASS' if passed else '❌ FAIL'}: {mean_score:.3f} vs threshold {threshold}"
        })

    return results


def define_gating_criteria(check_results):
    """
    🧠 YOUR TURN: Implement the gating logic.

    This is the DECISION ENGINE of the evaluation pipeline. Given raw
    check results, it must return a clear verdict: PASS, WARN, or FAIL.

    Args:
        check_results: list of dicts from run_stage_evaluation()
            Each dict has:
              - "passed": bool       — did it meet the threshold?
              - "action": str        — "BLOCK merge/release" or "WARN — review required"
              - "metric": str        — which metric was checked
              - "component": str     — which component was checked
              - "mean_score": float  — actual score achieved
              - "threshold": float   — minimum required score

    Returns:
        dict with:
          - "status": "PASS" | "WARN" | "FAIL"
          - "reasons": list of strings explaining the decision

    DESIGN GUIDELINES (implement your own version):
      - A single BLOCK failure → FAIL (non-negotiable hard gates)
      - A single WARN failure → WARN (needs human review)
      - Multiple WARN failures → consider escalating to FAIL
        (one soft issue = concern; three soft issues = systemic problem)
      - All checks pass → PASS
    """
    # ╔══════════════════════════════════════════════════╗
    # ║  🧠 YOUR TURN: Implement gating logic below     ║
    # ╚══════════════════════════════════════════════════╝

    status = "PASS"  # Start optimistic — downgrade as we find problems
    reasons = []

    # TODO: Loop through check_results
    # TODO: For each failed check, look at the "action" field:
    #       - If action contains "BLOCK" → set status to FAIL
    #       - If action contains "WARN" → set status to at least WARN
    # TODO: Count failed WARN checks — if >= 3, escalate to FAIL
    # TODO: Collect reasons explaining each failure
    # TODO: If everything passed, add a reason like "All checks passed"

    # YOUR CODE HERE
    pass

    return {"status": status, "reasons": reasons}

# ── Test your implementation ──
# This runs ONLY the CI stage so you can iterate quickly.
# Once your gating logic works here, it'll also work for pre-prod.
print("Testing gating criteria on CI stage...\n")
ci_results = run_stage_evaluation(evaluation_stages["ci"], df)

# Show each individual check result — does each look correct?
for r in ci_results:
    print(f"  {r['detail']}")

# Show the overall gate decision — did YOUR logic produce the right verdict?
gate_result = define_gating_criteria(ci_results)
print(f"\n🚦 CI Gate Decision: {gate_result['status']}")
for reason in gate_result.get("reasons", []):
    print(f"   → {reason}")

# Sanity check: if you see "PASS" with no reasons, you haven't
# implemented the logic yet. Go back and fill in the TODO section.

### Checkpoint ✅
After implementing `define_gating_criteria()`:
- The CI stage should show check results for all 4 CI checks
- Your gating function should return "PASS", "WARN", or "FAIL" with clear reasons
- If you see "PASS" with no reasons, you haven't implemented the logic yet!

**Don't move on until your gating function handles at least:** one BLOCK failure → FAIL, all pass → PASS, and WARN failures → WARN.

### 🔍 What Just Happened?

You built the **decision engine** of your evaluation pipeline. This is the piece that automates the judgment call: "Should we proceed?" In production systems, this gating logic runs automatically — a failed BLOCK check literally prevents code from merging or a release from deploying.

Now let's wire everything together into a complete pipeline. ⬇️

---

## Task 6: Implement the Full Evaluation Pipeline 🔗

### Goal
Wire together the stage evaluations into a single pipeline that runs CI and pre-production checks for a model version.

### Why a Pipeline (Not Just Individual Checks)?
Running checks individually is fine for debugging, but a **pipeline** gives you:
- A single command to evaluate everything
- A structured report you can log and audit
- Sequential gating (if CI fails, you might skip the expensive pre-prod check)
- A timestamp and metadata for reproducibility

> 🎯 **Why This Matters:** In real CI/CD systems, this pipeline runs automatically on every pull request. The report becomes the evidence for "why did we deploy (or not deploy) this version?"

### Steps
Run the cell below, which uses `run_stage_evaluation()` and your `define_gating_criteria()` function to build the pipeline.

In [ ]:
# ============================================================
# Full Evaluation Pipeline
# ============================================================
# This function wires together run_stage_evaluation() and
# define_gating_criteria() into a single end-to-end pipeline.
#
# PIPELINE ARCHITECTURE:
#   1. Run CI checks first (fast, catch breakages)
#   2. Run Pre-Production checks (thorough, catch regressions)
#   3. Collect results into a structured, auditable report
#
# IN A REAL SYSTEM this function would:
#   - Be triggered by CI/CD (GitHub Actions, Jenkins, etc.)
#   - Log the report to a database for historical tracking
#   - Post the summary as a PR comment
#   - Optionally SHORT-CIRCUIT if CI fails (skip expensive pre-prod)
#
# We don't short-circuit here because we want the full picture
# for learning purposes — but in production, failing CI early
# saves expensive pre-production compute.

def run_evaluation_pipeline(eval_df, stages_config):
    """
    Run the full evaluation pipeline across CI and pre-production stages.

    Args:
        eval_df: DataFrame with evaluation data (v2.0 and v2.1 scores)
        stages_config: dict of stage definitions (from evaluation_stages)

    Returns:
        pipeline_report: dict with metadata and per-stage results.
        This report is the SINGLE SOURCE OF TRUTH for "what happened
        during this evaluation run." It should be stored and versioned.
    """
    # Build the report header with metadata for traceability.
    # In a real system, you'd also include: git SHA, model artifact hash,
    # test suite version, and the evaluator identity.
    pipeline_report = {
        "timestamp": datetime.now().isoformat(),
        "model_version": "v2.1",
        "baseline_version": "v2.0",
        "total_test_cases": len(eval_df),
        "stages": {}
    }

    # Run each stage sequentially: CI first, then pre-production.
    # Order matters: CI is designed to be fast, pre-prod is thorough.
    for stage_key in ["ci", "pre_production"]:
        stage_config = stages_config[stage_key]

        # Step 1: Run all checks for this stage
        # This produces raw results (pass/fail per check)
        check_results = run_stage_evaluation(stage_config, eval_df)

        # Step 2: Apply YOUR gating criteria to decide PASS/WARN/FAIL
        # This is where the human judgment lives (encoded in your function)
        gate_decision = define_gating_criteria(check_results)

        # Step 3: Store everything in the report for audit trail
        pipeline_report["stages"][stage_key] = {
            "name": stage_config["name"],
            "checks": check_results,
            "decision": gate_decision
        }

        # Step 4: Print a summary (in production, this would be logged)
        # We continue running ALL stages even if one fails —
        # we want the complete picture, not just the first failure.
        status_icon = {"PASS": "✅", "WARN": "⚠️", "FAIL": "❌"}.get(gate_decision["status"], "❓")
        print(f"\n{'='*60}")
        print(f"{status_icon} Stage: {stage_config['name']} → {gate_decision['status']}")
        print(f"{'='*60}")
        for r in check_results:
            print(f"  {r['detail']}")
        if gate_decision["reasons"]:
            print(f"\n  Decision reasons:")
            for reason in gate_decision["reasons"]:
                print(f"    → {reason}")

    return pipeline_report

# ── Run the full pipeline! ──
# This is the moment of truth: does v2.1 pass your quality gates?
print("🚀 Running evaluation pipeline for LegalBot v2.1...\n")
pipeline_report = run_evaluation_pipeline(df, evaluation_stages)

### Checkpoint ✅
You should see the pipeline run both stages:
- **CI stage**: Results for 4 checks with a gate decision
- **Pre-production stage**: Results for 9 checks with a gate decision

If you see "PASS" with no reasons for both stages, go back and implement `define_gating_criteria()` properly.

### 🔍 What Just Happened?

You just ran your first **end-to-end evaluation pipeline**! The pipeline sequentially executed CI and pre-production checks, applied your gating logic, and produced a structured report. In a real system, this would run automatically in your CI/CD infrastructure (e.g., GitHub Actions, Jenkins) and the report would be attached to the pull request.

Next, we'll build a **readiness scorecard** — the stakeholder-friendly summary of these results. ⬇️

---

## Task 7: 🧠 Your Turn — Create a Readiness Scorecard 📋

### Goal
Create a **readiness scorecard** — a structured document that summarizes evaluation results and supports a go/no-go decision. This is what you'd show to leadership.

### Why Scorecards?
Engineers think in metrics and code. Executives think in risk and readiness. A scorecard **bridges the gap** — it translates your technical evaluation into a format that supports business decisions.

```
┌──────────────────────────────────────────────────────┐
│              READINESS SCORECARD                     │
│                                                      │
│  Overall Status:  🟡 CONDITIONAL                     │
│                                                      │
│  ┌────────────────┬────────┬────────┬───────┐       │
│  │ Component      │ v2.0   │ v2.1   │ Delta │       │
│  ├────────────────┼────────┼────────┼───────┤       │
│  │ Structured     │  0.89  │  0.90  │ +0.01 │       │
│  │ RAG            │  0.81  │  0.81  │  0.00 │       │
│  │ Generation     │  0.86  │  0.88  │ +0.02 │       │
│  └────────────────┴────────┴────────┴───────┘       │
│                                                      │
│  Regressions: 8 found (1 critical, 3 moderate)      │
│  Risks: Safety regression on ambiguous inputs        │
│  Recommendation: Deploy with guardrails              │
└──────────────────────────────────────────────────────┘
```

> 💡 **Key Insight:** The scorecard should give a decision-maker everything they need in 30 seconds: status, risks, and recommendation. Details are available if they want to dig in, but the top-level view must be instantly clear.

### 🧠 Your Task

Complete the `create_readiness_scorecard()` function below. Your scorecard must include:

1. **Overall readiness status** (Ready / Conditional / Not Ready)
2. **Component-level summary** (how each component performed)
3. **Regression summary** (what got worse and how bad)
4. **Risk assessment** (what risks exist if we deploy)
5. **Recommendation** (go / no-go / conditional-go with mitigations)

**Hint:** Don't just check averages — look for specific regressions. A component can look "fine on average" while hiding a critical regression in one metric.

In [ ]:
# ============================================================
# Readiness Scorecard Builder
# ============================================================
# A scorecard is the "executive one-pager" — it translates raw
# evaluation metrics into a business decision artifact.
#
# WHO READS THIS: VP of Engineering, Product Manager, Legal Counsel.
# They don't care about individual test case scores. They care about:
#   - Can we deploy safely? (risk assessment)
#   - What are the tradeoffs? (improvement vs. regression)
#   - What do we need to do? (clear recommendation)
#
# SCORECARD COMPONENTS:
#   1. Overall status: READY / CONDITIONAL / NOT_READY
#   2. Component scores: how each subsystem performed
#   3. Regressions: what got worse (and how bad)
#   4. Risks: what could go wrong if we deploy
#   5. Recommendation: concrete go/no-go with conditions

def create_readiness_scorecard(eval_df, pipeline_report):
    """
    🧠 YOUR TURN: Build a readiness scorecard.

    This function takes raw evaluation data and pipeline results
    and produces a structured decision document.

    Args:
        eval_df: DataFrame with all test case results
        pipeline_report: dict from run_evaluation_pipeline()

    Returns:
        dict with:
          - "overall_status": "READY" | "CONDITIONAL" | "NOT_READY"
          - "component_scores": per-component performance summary
          - "regressions": list of specific regressions found
          - "risks": list of deployment risks
          - "recommendation": actionable recommendation string

    DECISION GUIDELINES:
      READY       = All gates passed, no critical regressions,
                    no safety concerns
      CONDITIONAL = Gates passed, but concerning regressions exist.
                    Deploy only WITH specific mitigations.
      NOT_READY   = Gates failed OR critical safety regressions.
                    Must fix and re-evaluate.
    """
    # ╔══════════════════════════════════════════════════╗
    # ║  🧠 YOUR TURN: Build the scorecard below        ║
    # ╚══════════════════════════════════════════════════╝

    scorecard = {
        "model_version": "v2.1",
        "baseline": "v2.0",
        "evaluation_date": datetime.now().strftime("%Y-%m-%d"),
        "overall_status": "TODO",       # Set this: READY, CONDITIONAL, or NOT_READY
        "component_scores": {},          # Fill per-component summaries
        "regressions": [],               # List specific regressions found
        "risks": [],                     # List deployment risks
        "recommendation": "TODO"         # Write your recommendation
    }

    # ── Step 1: Calculate component-level scores ──────────────
    # Group by component, compute v2.0 mean, v2.1 mean, and delta
    # This gives the "big picture" per subsystem.
    #
    # Hint: eval_df.groupby("component")[["v2_0_score", "v2_1_score"]].mean()
    # Then compute delta as v2.1 - v2.0 for each component.
    # Store as: {"structured": "v2.0=0.89, v2.1=0.91, Δ=+0.02", ...}

    # TODO: YOUR CODE HERE

    # ── Step 2: Find all regressions ──────────────────────────
    # Filter the DataFrame where delta < 0 (v2.1 scored worse).
    # For each regression, include:
    #   - test_id, component, metric, description
    #   - magnitude (how much it dropped)
    #   - severity (CRITICAL if >0.05, MODERATE if >0.03, MINOR otherwise)
    #
    # Hint: regressed_rows = eval_df[eval_df["delta"] < 0]
    #       Then iterate and build a list of descriptive strings.

    # TODO: YOUR CODE HERE

    # ── Step 3: Assess deployment risks ───────────────────────
    # Not all regressions carry equal risk. Prioritize:
    #   - Safety regressions (HIGH RISK even if small magnitude)
    #   - Faithfulness regressions (hallucination risk)
    #   - Schema regressions (downstream pipeline breakage)
    #   - Edge case regressions (OCR noise, ambiguous inputs)
    #
    # Each risk should explain: WHAT could go wrong, HOW LIKELY,
    # and WHAT THE IMPACT would be.

    # TODO: YOUR CODE HERE

    # ── Step 4: Set overall status ────────────────────────────
    # Apply the decision guidelines from the docstring above.
    # Check gate results from pipeline_report["stages"][stage]["decision"]["status"]

    # TODO: YOUR CODE HERE

    # ── Step 5: Write a clear recommendation ──────────────────
    # Be specific and actionable. Bad: "Maybe deploy with caution."
    # Good: "Deploy with enhanced monitoring on multi-doc synthesis
    #        queries and a 0.90 safety threshold circuit breaker."

    # TODO: YOUR CODE HERE

    return scorecard

# ── Generate and display the scorecard ──
scorecard = create_readiness_scorecard(df, pipeline_report)

print("=" * 60)
print("📋 READINESS SCORECARD: LegalBot v2.1")
print("=" * 60)
print(f"Date: {scorecard['evaluation_date']}")
print(f"Version: {scorecard['model_version']} (baseline: {scorecard['baseline']})")
print(f"\n🔴🟡🟢 Overall Status: {scorecard['overall_status']}")

if scorecard.get("component_scores"):
    print(f"\n{'─'*40}")
    print("Component Scores:")
    for comp, summary in scorecard["component_scores"].items():
        print(f"  {comp}: {summary}")

if scorecard.get("regressions"):
    print(f"\n{'─'*40}")
    print(f"Regressions Found: {len(scorecard['regressions'])}")
    for reg in scorecard["regressions"]:
        print(f"  ⬇️  {reg}")

if scorecard.get("risks"):
    print(f"\n{'─'*40}")
    print("Risks:")
    for risk in scorecard["risks"]:
        print(f"  ⚠️  {risk}")

print(f"\n{'─'*40}")
print(f"Recommendation: {scorecard['recommendation']}")

### Checkpoint ✅
Your scorecard should:
- Have a non-"TODO" overall status
- List at least 3 component scores
- Identify multiple regressions (there are several in the data!)
- Include at least one risk
- Have a clear recommendation

If your scorecard still shows "TODO" values, you haven't completed the implementation.

### 🔍 What Just Happened?

You just built a **readiness scorecard** — the single most important artifact in an evaluation pipeline. This is the document that turns raw metrics into a decision. In practice, scorecards are versioned, stored, and referenced in post-mortems and launch reviews.

Now let's take a deeper look at the regressions to understand their severity and patterns. ⬇️

---

## Task 8: Run Regression Detection Across Versions 🔎

### Goal
Now perform a systematic regression analysis comparing v2.0 and v2.1 across all test cases.

### Why Regression Detection?
A model version can look "better on average" while being **worse on specific, critical test cases**. Regression detection digs below the averages to find those trouble spots.

```
  Regression Detection Flow:
  ┌─────────────┐    ┌──────────────┐    ┌─────────────┐
  │ Compare each │───▶│ Flag where   │───▶│ Sort by     │
  │ test case    │    │ v2.1 < v2.0  │    │ severity    │
  │ v2.0 vs v2.1│    │              │    │ CRITICAL >  │
  └─────────────┘    └──────────────┘    │ MODERATE >  │
                                          │ MINOR       │
                                          └─────────────┘
```

> 🎯 **Why This Matters:** The most dangerous regressions are the ones that hide in averaging. If 8 out of 10 test cases improved, the average looks great — but those 2 regressions might be on safety-critical scenarios.

### Steps
Run the cell below to identify and categorize all regressions.

In [ ]:
# ============================================================
# Regression Detection: v2.0 → v2.1
# ============================================================
# WHY SEPARATE REGRESSION DETECTION FROM THE PIPELINE?
# The pipeline checks aggregate metrics (averages above thresholds).
# Regression detection looks at INDIVIDUAL test cases — it finds
# the specific scenarios where v2.1 is worse, even if the averages
# look good.
#
# ANALOGY: The pipeline is a health check that says "your blood
# pressure is normal." Regression detection is the specialist who
# says "but this specific artery has a blockage."
#
# SEVERITY CLASSIFICATION:
#   CRITICAL (>5 point drop): Something fundamentally broke.
#     Example: 0.84 → 0.79 on faithfulness = 5-point drop.
#   MODERATE (>3 point drop): Concerning, needs investigation.
#     Also: ANY safety-related regression, regardless of magnitude.
#   MINOR (<3 point drop): Small and possibly within noise margin.
#     Still worth tracking but unlikely to block a release.

def detect_regressions(eval_df, significance_threshold=0.03):
    """
    Identify and classify regressions between v2.0 and v2.1.

    Scans every test case for score decreases and classifies them
    by severity. Safety regressions are automatically promoted to
    at least MODERATE severity (even if the magnitude is small).

    Args:
        eval_df: DataFrame with v2_0_score, v2_1_score, and delta columns
        significance_threshold: minimum delta to classify as MODERATE
                                (default: 0.03 = 3 percentage points)

    Returns:
        List of regression dicts, sorted worst-first (most negative delta).
        Each dict includes: test_id, component, metric, description,
        v2_0 score, v2_1 score, delta, and severity classification.
    """
    regressions = []

    for _, row in eval_df.iterrows():
        # Only look at cases where v2.1 performed WORSE (negative delta)
        if row["delta"] < 0:
            # Classify severity by how much the score dropped
            severity = (
                "CRITICAL" if abs(row["delta"]) > 0.05     # > 5 points = alarm bells
                else "MODERATE" if abs(row["delta"]) > significance_threshold  # > 3 points
                else "MINOR"                                 # small drop, may be noise
            )

            # SPECIAL RULE: Safety regressions are NEVER "minor."
            # In a legal AI system, even a 1-2% drop in safety
            # could mean more hallucinated legal advice reaching users.
            # Promoting to MODERATE ensures it gets flagged in reviews.
            if row["metric"] == "safety" and severity == "MINOR":
                severity = "MODERATE"

            regressions.append({
                "test_id": row["test_id"],
                "component": row["component"],
                "metric": row["metric"],
                "description": row["description"],
                "v2_0": row["v2_0_score"],
                "v2_1": row["v2_1_score"],
                "delta": round(row["delta"], 3),
                "severity": severity
            })

    # Sort by delta ascending (most negative first) so the worst
    # regressions appear at the top of the report. This follows the
    # "bad news first" principle: decision-makers should see the
    # biggest problems before the minor ones.
    return sorted(regressions, key=lambda x: x["delta"])

# ── Run regression detection ──
regressions = detect_regressions(df)

# Display results as a compact table for quick scanning
print(f"🔍 Found {len(regressions)} regressions in v2.1 vs v2.0\n")
print(f"{'Test ID':<10} {'Component':<14} {'Metric':<22} {'Delta':<8} {'Severity':<10}")
print("─" * 64)
for reg in regressions:
    print(f"{reg['test_id']:<10} {reg['component']:<14} {reg['metric']:<22} {reg['delta']:<8} {reg['severity']:<10}")

In [ ]:
# ============================================================
# Regression Categorization: Patterns and Priorities
# ============================================================
# After detecting individual regressions, we need to find PATTERNS.
# Random, unrelated regressions are concerning but manageable.
# SYSTEMATIC regressions (all in one component, all on complex inputs)
# suggest an architectural problem that needs deeper investigation.

from collections import Counter

# Count regressions by severity level
# This tells us: is the problem mostly minor noise, or are there
# critical/moderate issues that need immediate attention?
severity_counts = Counter(r["severity"] for r in regressions)

# Count regressions by component
# This tells us: is the problem concentrated in one subsystem,
# or spread across the whole system?
component_counts = Counter(r["component"] for r in regressions)

print("\n📊 Regression Summary")
print(f"   By severity: {dict(severity_counts)}")
print(f"   By component: {dict(component_counts)}")

# ── Highlight the most concerning regressions ──
# These are the ones that should be discussed first in a review meeting.

# Critical regressions: > 5 point drops. These are potential blockers.
critical = [r for r in regressions if r["severity"] == "CRITICAL"]

# Safety regressions: ANY drop in safety metrics, regardless of magnitude.
# In a legal context, safety = no hallucinated legal advice.
safety_regs = [r for r in regressions if r["metric"] == "safety"]

if critical:
    print(f"\n🚨 CRITICAL regressions ({len(critical)}):")
    print("   These dropped by more than 5 percentage points — investigate immediately.")
    for c in critical:
        print(f"   {c['test_id']}: {c['description']}")
        print(f"           {c['metric']} dropped {c['delta']} ({c['v2_0']:.2f} → {c['v2_1']:.2f})")

if safety_regs:
    print(f"\n⚠️  Safety regressions ({len(safety_regs)}):")
    print("   ANY safety drop in a legal system demands investigation.")
    for s in safety_regs:
        print(f"   {s['test_id']}: {s['description']}")
        print(f"           safety dropped {s['delta']} ({s['v2_0']:.2f} → {s['v2_1']:.2f})")

# ── Look for the pattern ──
# Hint to the learner: what do the regressed test cases have in common?
print("\n💡 LOOK FOR THE PATTERN:")
print("   Review the descriptions of regressed test cases above.")
print("   What do they have in common? (Noisy inputs? Complex queries? Cross-references?)")
print("   Understanding the pattern tells you WHAT CHANGED in v2.1.")

### Checkpoint ✅
You should see:
- **7–8 regressions** total (the exact number depends on the dataset)
- At least one **CRITICAL** regression
- At least one **safety** regression
- Regressions spread across all three components

Note the pattern: many metrics improved, but specific areas regressed — especially on complex/edge-case inputs.

### 🔍 What Just Happened?

You ran a **systematic regression analysis** and discovered a key pattern: v2.1 generally improves on standard/simple cases but regresses on complex, noisy, or edge-case inputs. This is a classic pattern when models are fine-tuned — they optimize for the common case at the expense of the tail. The safety regressions are especially concerning.

Now it's time for the most important task: interpreting these results and making a decision. ⬇️

---

## Task 9: 🧠 Your Turn — Interpret Regressions and Recommend

### Goal
Based on the regression analysis, make a concrete launch recommendation for v2.1.

### Why This Task Matters Most
Everything you've built so far — the pipeline, the scorecard, the regression detection — exists to support **this moment**: the launch decision. In the real world, someone has to look at the data and say "go" or "no-go." Frameworks and automation help, but the final judgment is yours.

> 🎯 **Why This Matters:** There is no "right" answer that avoids all risk. Deploying v2.1 means accepting the regressions. Not deploying means missing the improvements. Your job is to make the **most informed, defensible** call.

### 🧠 Your Task

This is the most important cell in the lab. Write your analysis and recommendation below.

Answer these questions:
1. **What pattern do you see in the regressions?** (Are they random or systematic?)
2. **Which regression concerns you most and why?**
3. **Would you recommend deploying v2.1?** (YES / YES WITH CONDITIONS / NO)
4. **If YES WITH CONDITIONS, what are the conditions?**
5. **If NO, what must be fixed before re-evaluation?**

**Hint:** Look at the *types* of test cases that regressed, not just the metrics. OCR noise, multi-document synthesis, cross-references, ambiguous inputs — what do these have in common?

### Checkpoint ✅
You should have:
- Identified a clear pattern in the regressions (not just "some things got worse")
- Named a specific most-concerning regression with reasoning
- Made a concrete decision (not wishy-washy "it depends")
- If conditional, stated specific, actionable conditions

### 🔍 What Just Happened?

You made your first **evidence-based launch recommendation**. The key skill here isn't picking the "right" answer — it's articulating your reasoning so clearly that others can evaluate your logic, even if they disagree with your conclusion.

Now let's tackle an even trickier challenge: what do you do when metrics contradict each other? ⬇️

---

## Task 10: ⚠️ Open Challenge — Conflicting Metrics

### Goal
Handle a realistic scenario where **metrics conflict**: one metric improved significantly while a related metric regressed.

### Why This Is Hard
In the real world, metrics rarely all move in the same direction. Improvements in one area often come at the cost of regressions in another. **This is the hardest kind of evaluation decision** because there's no simple rule to follow.

> ⚠️ **Common Mistake:** Treating all metric movements as independent. If faithfulness improved on simple questions but regressed on complex ones, that's not two unrelated events — it's one architectural change with different effects.

### Scenario
Look at the RAG component data. You'll notice:
- **Faithfulness on single-document questions** improved (v2.0: 0.92 → v2.1: 0.94) ✅
- **Faithfulness on multi-document synthesis** regressed (v2.0: 0.84 → v2.1: 0.79) ❌

Similarly in generation:
- **Coherence** improved significantly across the board ✅
- **Completeness on cross-references** regressed (v2.0: 0.77 → v2.1: 0.72) ❌
- **Safety on ambiguous questions** regressed (v2.0: 0.88 → v2.1: 0.85) ❌

### ⚠️ Your Task

This is an **open challenge** — there are multiple valid approaches.

1. **Diagnose the conflict:** Why might these metrics move in opposite directions? What might have changed in v2.1 to cause this?
2. **Propose a resolution framework:** When metrics conflict, how should an evaluation lead decide? Write a decision framework (3–5 rules).
3. **Apply your framework:** Use your rules to make a call on the RAG faithfulness conflict above.
4. **Propose a mitigation:** If you deploy v2.1 despite the regression, what guardrail or monitoring would you add?

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  ⚠️ OPEN CHALLENGE: Conflicting Metrics Visualization          ║
# ╚══════════════════════════════════════════════════════════════════╝
# This cell makes the metric conflicts VISIBLE so you can reason
# about them systematically.
#
# WHY CONFLICTS HAPPEN:
# When a model is updated (fine-tuned, retrained, or re-prompted),
# improvements in one area often come at the cost of regressions
# in another. Common causes:
#   - New embeddings optimize for common queries but lose rare ones
#   - Shortened context windows produce more coherent BUT less complete output
#   - Safety-tuning makes the model more cautious on clear cases
#     but MORE confused on ambiguous ones
#   - Better chunking helps single-doc retrieval but hurts cross-doc

# ── Conflict 1: RAG Faithfulness ──
# Same metric (faithfulness), but opposite results depending on
# query complexity. This is a CLASSIC fine-tuning tradeoff.
print("RAG Faithfulness — Conflicting results by query complexity:")
print("(Same metric, opposite direction depending on input type)\n")
rag_faith = df[(df["component"] == "rag") & (df["metric"] == "faithfulness")]
for _, row in rag_faith.iterrows():
    direction = "📈" if row["delta"] > 0 else "📉" if row["delta"] < 0 else "➡️"
    print(f"  {direction} {row['test_id']}: {row['description']}")
    print(f"     v2.0: {row['v2_0_score']:.2f} → v2.1: {row['v2_1_score']:.2f} (Δ {row['delta']:+.2f})")

# ── Conflict 2: Generation Quality vs. Safety ──
# Different metrics in the same component pulling in opposite directions.
# v2.1 writes BETTER text (more coherent, better tone) but MISSES things
# (less complete cross-references) and has SAFETY regressions.
# This suggests: the model learned to write "prettier" output at the
# cost of thoroughness and safety guardrails.
print(f"\n{'─'*50}")
print(f"\nGeneration — Coherence vs Completeness vs Safety:")
print("(Different metrics, opposite directions — a quality/safety tradeoff)\n")
gen = df[df["component"] == "generation"]
for _, row in gen.iterrows():
    direction = "📈" if row["delta"] > 0 else "📉" if row["delta"] < 0 else "➡️"
    print(f"  {direction} {row['test_id']}: [{row['metric']}] {row['description']}")
    print(f"     v2.0: {row['v2_0_score']:.2f} → v2.1: {row['v2_1_score']:.2f} (Δ {row['delta']:+.2f})")

print(f"\n💡 KEY QUESTION: When coherence goes up but safety goes down,")
print(f"   which metric should 'win'? Your decision framework (next cell)")
print(f"   should answer this.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  ⚠️ YOUR ANSWERS: Conflicting Metrics Framework                ║
# ╚══════════════════════════════════════════════════════════════════╝
# These questions don't have a single correct answer.
# The goal is to develop a SYSTEMATIC, REPEATABLE way of handling
# metric conflicts — not just "gut feeling" decisions.

# 1. Why might these metrics move in opposite directions?
# ────────────────────────────────────────────────────────
# Think architecturally: what ONE change could cause BOTH effects?
# Possibilities to consider:
#   - New embedding model with different strengths/weaknesses
#   - Modified chunking strategy (smaller chunks = more precise but less context)
#   - Updated prompt template (more structured → more coherent but less flexible)
#   - Fine-tuning on examples that are mostly "simple" cases
#   - Prompt engineering that over-optimizes for the common case
conflict_diagnosis = """
YOUR ANALYSIS HERE
"""

# 2. Your decision framework for conflicting metrics (3-5 rules)
# ────────────────────────────────────────────────────────
# A good framework is one that ANOTHER engineer could apply
# consistently and reach the same conclusion you would.
# Consider ranking metrics by:
#   - Safety impact (safety always wins over quality)
#   - Frequency of affected scenarios (rare edge case vs. common use)
#   - Reversibility of harm (wrong legal advice vs. awkward phrasing)
#   - Magnitude of change (2% vs. 10%)
#   - Whether monitoring/guardrails can mitigate the regression
decision_framework = """
YOUR FRAMEWORK HERE

Rule 1:
Rule 2:
Rule 3:
"""

# 3. Apply your framework to the RAG faithfulness conflict
# ────────────────────────────────────────────────────────
# Specific conflict: faithfulness improved on single-doc (0.92→0.94)
#                    faithfulness REGRESSED on multi-doc (0.84→0.79)
# Use your rules. Be explicit about WHICH rule drives the decision.
# Show your work: "Applying Rule 2, since multi-doc queries
# represent ~30% of production traffic, and the 5-point drop
# exceeds our CRITICAL threshold..."
framework_application = """
YOUR APPLICATION HERE
"""

# 4. Proposed mitigation if deploying despite regression
# ────────────────────────────────────────────────────────
# "Monitor it" is NOT a mitigation. A mitigation is a SPECIFIC
# action that reduces the probability or impact of the regression.
# Examples of GOOD mitigations:
#   - Route multi-doc queries to v2.0 via a feature flag
#   - Add a confidence threshold: if < 0.7, return "I don't know"
#   - Implement a dual-model check: run both v2.0 and v2.1, compare
#   - Add a guardrail that detects cross-doc synthesis and adds disclaimers
proposed_mitigation = """
YOUR MITIGATION HERE
"""

print("⚠️  Open challenge answers recorded.")
print("Your instructor will lead a discussion comparing approaches.")

### Checkpoint ✅
You should have written substantive answers for all four parts. Strong answers will:
- Recognize that improvements on simple cases + regressions on complex cases suggests an architectural change (e.g., new embedding model, modified chunking, different prompt template)
- Include a framework that handles both severity and scope (a small regression on many cases vs. a large regression on few cases)
- Apply the framework consistently
- Propose a concrete, implementable mitigation (not just "monitor it")

### 🔍 What Just Happened?

You tackled the hardest evaluation problem: **conflicting metrics**. This is where evaluation becomes more art than science — you need frameworks and judgment, not just thresholds. The decision framework you wrote here is something you'll use in every future model evaluation.

Almost done! Let's compile everything into a final presentation-ready report. ⬇️

---

## Task 11: Present Readiness Decision with Evidence 🎤

### Goal
Compile your analysis into a final readiness presentation. This is the deliverable you'd present to stakeholders — the culmination of the entire evaluation pipeline.

### Why This Is the Deliverable
Everything you've built — the pipeline, the scorecard, the regression analysis — is in service of this moment. A well-structured readiness decision gives leadership the confidence to say "go" or the evidence to say "not yet."

> 💡 **Key Insight:** An evaluation pipeline that produces great data but no clear recommendation has failed its purpose. The whole point is to turn data into decisions.

### Steps
Run the cell below to generate the final pipeline summary, then write your executive decision.

In [ ]:
# ============================================================
# Final Pipeline Summary Report
# ============================================================
# This cell produces the "executive dashboard" — everything a
# decision-maker needs to see on one screen.
#
# STRUCTURE:
#   Section 1: Overall performance comparison by component (the "big picture")
#   Section 2: Gate results from CI and pre-prod (did we pass or fail?)
#   Section 3: Regression breakdown by severity (how bad is it?)
#   Section 4: Notable improvements (what got better — for balance)
#
# WHY INCLUDE IMPROVEMENTS? It's tempting to only show problems,
# but decision-makers need the full picture. If you only show
# regressions, they'll never approve any deployment.

print("=" * 70)
print("📋 LEGALBOT v2.1 — EVALUATION PIPELINE FINAL REPORT")
print("=" * 70)

# ── Section 1: Component-level overview ──
# "Did each subsystem get better or worse overall?"
# This is the 5-second summary that frames everything else.
print("\n📊 Overall Performance Comparison (by component):")
for component in ["structured", "rag", "generation"]:
    comp_data = df[df["component"] == component]
    v20_mean = comp_data["v2_0_score"].mean()
    v21_mean = comp_data["v2_1_score"].mean()
    delta = v21_mean - v20_mean
    direction = "📈" if delta > 0 else "📉" if delta < 0 else "➡️"
    print(f"  {direction} {component.upper():>12}: v2.0={v20_mean:.3f} → v2.1={v21_mean:.3f} (Δ {delta:+.3f})")

# ── Section 2: Gate results ──
# "Did each stage of the pipeline pass?"
# This is the clear yes/no that leadership needs.
print(f"\n🚦 Stage Gate Results:")
for stage_key, stage_data in pipeline_report.get("stages", {}).items():
    status = stage_data["decision"]["status"]
    icon = {"PASS": "✅", "WARN": "⚠️", "FAIL": "❌"}.get(status, "❓")
    print(f"  {icon} {stage_data['name']}: {status}")
    # If there are reasons, show them indented
    for reason in stage_data["decision"].get("reasons", []):
        print(f"       → {reason}")

# ── Section 3: Regression severity breakdown ──
# "How many things got worse, and how serious is it?"
# The severity distribution tells you whether this is a minor
# cleanup or a major problem.
print(f"\n⬇️ Regressions: {len(regressions)} total")
print(f"   CRITICAL: {severity_counts.get('CRITICAL', 0)} — dropped >5 points, investigate immediately")
print(f"   MODERATE: {severity_counts.get('MODERATE', 0)} — dropped >3 points or safety-related")
print(f"   MINOR:    {severity_counts.get('MINOR', 0)} — small drops, may be noise")

# ── Section 4: Key improvements (for balance) ──
# Only show improvements > 2 points to keep it concise.
# Sorted by delta descending so the biggest wins appear first.
improvements = df[df["delta"] > 0.02].sort_values("delta", ascending=False)
print(f"\n⬆️ Notable Improvements (Δ > +0.02): {len(improvements)}")
for _, imp in improvements.head(5).iterrows():
    print(f"   {imp['test_id']}: {imp['description']} — {imp['metric']} +{imp['delta']:.2f}")

print(f"\n{'='*70}")
print("Write your executive decision in the next cell ⬇️")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  FINAL DECISION: Write your executive summary                  ║
# ╚══════════════════════════════════════════════════════════════════╝
# This is the DELIVERABLE of the entire lab — everything else was
# building towards this moment.
#
# IMAGINE: You're printing this and handing it to your VP of
# Engineering in a meeting room. They have 2 minutes to read it.
#
# WRITING GUIDELINES:
#   - Lead with the decision (don't bury it at the bottom)
#   - Reference specific data (test IDs, metrics, deltas)
#   - Be honest about risks (don't hide the safety regression)
#   - Make recommendations actionable ("add X monitoring" not "be careful")
#   - Use plain language (not everyone knows what "faithfulness" means)

executive_summary = """
LEGALBOT v2.1 READINESS DECISION
=================================
Date: [TODAY'S DATE]
Evaluator: [YOUR NAME]
Decision: [READY / CONDITIONAL / NOT READY]

Evidence Summary:
-----------------
[Summarize the key findings — 3-5 bullet points]
[Reference specific test IDs and scores, e.g.:]
[  - R-006: Multi-doc faithfulness dropped 0.84→0.79 (CRITICAL)  ]
[  - G-007: Safety regression on unauthorized interpretations      ]
[  - 22 of 31 test cases improved in v2.1                         ]

Risks:
------
[List the key risks in plain language — 2-3 items]
[  - Higher chance of hallucinated cross-references in legal advice ]
[  - Safety regression on ambiguous questions (0.88→0.85)           ]
[  - OCR-degraded documents handle worse (0.71→0.68)                ]

Conditions (if conditional):
-----------------------------
[What must be TRUE for this to deploy safely?]
[Be specific — "add monitoring" is not enough. What EXACTLY?]
[  - Route multi-doc synthesis queries through v2.0 fallback       ]
[  - Deploy 15-minute safety monitoring with auto-circuit-breaker  ]
[  - Fix OCR regression before next release (track in JIRA-XXX)    ]

Recommended Next Steps:
-----------------------
[What should happen in the next 48 hours?]
[Who needs to do what, and by when?]
[  - Engineering: implement multi-doc fallback route (2 days)      ]
[  - QA: expand OCR test suite before v2.2                         ]
[  - Legal: review safety regression with compliance team          ]
"""

print(executive_summary)

### Checkpoint ✅
Your executive summary should:
- State a clear decision (not "maybe" or "it depends")
- Reference specific evidence from the evaluation (test IDs, metrics, deltas)
- Acknowledge risks honestly
- Propose actionable next steps
- Be written for a non-technical audience (executives, product managers)

### 🔍 What Just Happened?

You just produced the **final deliverable** of an end-to-end evaluation pipeline: an evidence-backed launch decision. This is the artifact that gets attached to the deployment record and referenced in future post-mortems. Congratulations! 🎉

---

## Validation / Success Criteria

You have successfully completed this lab if:

✅ You loaded and explored the multi-component evaluation dataset  
✅ You reasoned about evaluation stages before implementing them  
✅ You implemented working gating criteria with PASS/WARN/FAIL logic  
✅ You created a readiness scorecard with all required components  
✅ You identified and categorized regressions across model versions  
✅ You made a concrete deployment recommendation with evidence  
✅ You handled conflicting metrics systematically (not just intuitively)  
✅ You wrote an executive summary suitable for a stakeholder meeting  

---

## Troubleshooting

| Symptom | Likely Cause | Fix |
|---------|-------------|-----|
| `ModuleNotFoundError` for pandas/numpy | Install cell didn't run | Re-run the `pip install` cell and restart kernel |
| Gating criteria always returns "PASS" | `define_gating_criteria()` not implemented | Go back to Task 5 and implement the logic |
| Scorecard shows all "TODO" | `create_readiness_scorecard()` not implemented | Go back to Task 7 and implement the function |
| No regressions detected | Significance threshold too high | Check `detect_regressions()` — default threshold is 0.03 |
| Pipeline report is empty | Cells run out of order | Restart kernel and run all cells from the top |
| `KeyError` on column names | DataFrame not created | Ensure Task 2 data cell was executed |

---

## Stretch Goal / Extension

If you finish early, try one or both of these extensions:

### Extension 1: Automated Report Generator
Write a function that takes the pipeline report, regressions list, and scorecard and generates a formatted Markdown report string. Include tables, severity badges, and a traffic-light summary.

### Extension 2: Statistical Significance Testing
The current regression detection uses a simple threshold. Extend it to use bootstrapped confidence intervals: re-sample the test cases 1000 times and determine whether the observed regression is statistically significant or just noise.

---

## 🔑 Key Takeaways

1. 🏗️ **Evaluation is multi-staged** — different checks belong at CI, pre-production, and production, with different speed/thoroughness tradeoffs.

2. 🚦 **Gating criteria turn metrics into decisions** — without explicit gating logic, evaluation data sits unused. The gating function is the bridge between measurement and action.

3. 📋 **Readiness scorecards communicate evaluation to stakeholders** — the scorecard bridges the gap between engineering metrics and business decisions. If you can't summarize it for a non-technical audience, you don't understand it well enough.

4. 🔍 **Regressions hide in averages** — a component can look "improved overall" while harboring critical regressions on edge cases or complex inputs. Always look at individual test cases, not just aggregates.

5. ⚖️ **Conflicting metrics require frameworks, not intuition** — when one metric improves and another regresses, you need systematic rules to make the call. Document your framework so others can audit your reasoning.

6. 🏭 **The evaluation pipeline IS the product** — in production AI systems, the evaluation infrastructure is as important as the model itself. A great model with a bad evaluation pipeline is a liability.

---

## 🎓🎉 Course Complete! Congratulations!

You've completed all 8 modules of the **Advanced AI Model Evaluation** course. Here's a recap of your journey:

| Module | Topic | What You Learned |
|--------|-------|-----------------|
| 🔭 **Module 01** | Modern AI Evaluation Foundations | Evaluation dimensions, Langfuse traces, scores, rule/judge/human evaluators |
| 🔍 **Module 02** | Failure Mode Analysis | Failure taxonomy, embeddings + clustering, impact prioritization |
| 📋 **Module 03** | Evaluating Structured Outputs | Schema validation, weighted field scoring, consistency checks, go/no-go |
| 📚 **Module 04 / 04b** | Evaluating RAG Systems | Recall/Precision/nDCG, faithfulness (NLI), stress tests, building on a vector DB |
| ✍️ **Module 05** | Evaluating Generative Outputs | Claim-level factuality, completeness, numeric checks, LLM-as-judge rubrics |
| 🤖 **Module 06 / 06b** | Evaluating Agentic Systems | Tool selection, input validation, end-state grading, hard gates, pass@k vs pass^k |
| 🛡️ **Module 07 / 07b** | Robustness, Safety & Guardrails | Perturbations, layered safety evaluation, prompt-injection metrics and defenses |
| 🏁 **Module 08** | End-to-End Pipeline (this lab!) | Multi-stage pipelines, gating, scorecards, regression detection, launch decisions |

### 🚀 What's Next?

You now have the skills to:
- **Design** evaluation pipelines for any AI system
- **Implement** multi-stage evaluation with gating criteria
- **Detect** regressions that hide below surface-level metrics
- **Communicate** evaluation results to technical and non-technical stakeholders
- **Make** evidence-based launch decisions under uncertainty

> 💡 **Final Thought:** The best evaluation pipelines are living systems — they evolve as your models evolve, as your understanding of failure modes deepens, and as your users reveal new edge cases. What you built today is a foundation, not a finish line. Keep iterating! 🔄

---

## 🏋️ Exercises

Practice end-to-end pipeline evaluation: scorecards, gating, and regression tracking.

### Exercise 1 — Beginner: Weighted readiness score
Combine dimension scores with weights into one readiness number.

<details>
<summary>Show solution</summary>

```python
def readiness(scores, weights):
    return sum(scores[k]*weights[k] for k in scores) / sum(weights.values())
print(readiness({"acc":0.9,"safety":1.0}, {"acc":1,"safety":2}))
```
</details>

### Exercise 2 — Applied: Pass/fail gate
Fail launch if any dimension is below its threshold.

<details>
<summary>Show solution</summary>

```python
def gate(scores, thresholds):
    fails = [k for k in scores if scores[k] < thresholds[k]]
    return ("PASS" if not fails else "FAIL", fails)
print(gate({"acc":0.7,"safety":1.0}, {"acc":0.8,"safety":0.9}))
```
</details>

### Exercise 3 — Challenge: Regression detection
Flag dimensions that dropped >5% vs the previous run.

<details>
<summary>Show solution</summary>

```python
def regressions(prev, cur, tol=0.05):
    return [k for k in cur if prev.get(k,0) - cur[k] > tol]
print(regressions({"acc":0.9}, {"acc":0.83}))
```
</details>

### Exercise 4 — Stretch: Design a CI gate
Sketch how this scorecard would block a PR in CI, including which metrics are hard vs soft gates. (Open-ended.)
